# φ^∞ Resonance-Path HD Wallet Demonstration

This notebook demonstrates **BIP-32 compatible Hierarchical Deterministic (HD) key derivation** using TUPT modular arithmetic instead of ECDSA point multiplication.

**Key Properties:**
- Full BIP-32 path syntax (`m/44'/0'/0'/0/0`)
- Hardened and normal derivation modes
- TUPT-projected keys in $\mathbb{Z}_{12289}$
- TUPT M-of-N Multisig aggregation

In [ ]:
import secrets

import matplotlib.pyplot as plt

from phi_infinity_lattice_compression.bitcoin_extensions import (
    TUPT_MODULO,
    TUPTHDWallet,
    TUPTMultisig,
)

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=['#00f0ff', '#ffd700', '#ff3366', '#77dd77'])

## 1. Master Key Generation

Generate a master key from a 256-bit random seed, following BIP-32 structure with TUPT projection.

In [ ]:
seed = secrets.token_bytes(32)
master = TUPTHDWallet.from_seed(seed)

print(f"Seed (hex):           {seed.hex()}")
print(f"Master Private Key:   {master.key}")
print(f"Master Public Locus:  {master.public_locus}")
print(f"Master Chain Code:    {master.chain_code.hex()[:32]}...")
print(f"Depth:                {master.depth}")
print(f"Key in TUPT space:    0 ≤ {master.key} < {TUPT_MODULO} ✓")

## 2. BIP-44 Path Derivation Tree

We derive a full BIP-44 tree: `m / 44' / 0' / 0' / 0 / {0..19}` — the first 20 receiving addresses.

In [ ]:
# Derive the account node
account = master.derive_path("m/44'/0'/0'")
print("Account node (m/44'/0'/0'):")
print(f"  Key:    {account.key}")
print(f"  Locus:  {account.public_locus}")
print(f"  Depth:  {account.depth}")

# Derive external chain
external = account.derive_child(0)

# Derive 20 receiving addresses
addresses = []
print("\nFirst 20 receiving addresses (m/44'/0'/0'/0/i):")
print(f"{'Index':<8} {'Key':<10} {'Public Locus':<14} {'Digital Root':<14}")
print("-" * 48)
for i in range(20):
    child = external.derive_child(i)
    dr = child.key % 9
    dr = 9 if dr == 0 else dr
    addresses.append(child)
    print(f"{i:<8} {child.key:<10} {child.public_locus:<14} {dr:<14}")

## 3. Determinism Verification

Regenerating the same seed must produce identical keys at every tree level.

In [ ]:
# Regenerate from the same seed
master_b = TUPTHDWallet.from_seed(seed)
child_b = master_b.derive_path("m/44'/0'/0'/0/7")
child_a = master.derive_path("m/44'/0'/0'/0/7")

print(f"Original  m/44'/0'/0'/0/7 key: {child_a.key}")
print(f"Rederived m/44'/0'/0'/0/7 key: {child_b.key}")
print(f"\n✓ Deterministic: {child_a.key == child_b.key}")

# Different seed must produce different keys
different_seed = secrets.token_bytes(32)
master_c = TUPTHDWallet.from_seed(different_seed)
child_c = master_c.derive_path("m/44'/0'/0'/0/7")
print(f"\nDifferent seed key: {child_c.key}")
print(f"✓ Different from original: {child_a.key != child_c.key}")

## 4. Key Distribution Visualization

In [ ]:
# Generate 500 addresses from different seeds
all_keys = []
all_loci = []
for _ in range(25):
    s = secrets.token_bytes(32)
    m = TUPTHDWallet.from_seed(s)
    for j in range(20):
        c = m.derive_path(f"m/44'/0'/0'/0/{j}")
        all_keys.append(c.key)
        all_loci.append(c.public_locus)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(all_keys, bins=50, color='#00f0ff', edgecolor='#111', alpha=0.85)
axes[0].set_xlabel('Private Key Value', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('TUPT-HD Private Key Distribution (n=500)', fontsize=13)

axes[1].hist(all_loci, bins=50, color='#ffd700', edgecolor='#111', alpha=0.85)
axes[1].set_xlabel('Public Locus Value', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('TUPT-HD Public Locus Distribution (n=500)', fontsize=13)

plt.tight_layout()
plt.show()

## 5. TUPT Multisig Aggregation

Demonstrates M-of-N multisignature aggregation using φ-weighted modular summation.

In [ ]:
# Create a 2-of-4 multisig
signers = [TUPTHDWallet.from_seed(secrets.token_bytes(32)) for _ in range(4)]
public_loci = [s.public_locus for s in signers]

print("Signer public loci:")
for i, locus in enumerate(public_loci):
    print(f"  Signer {i}: {locus}")

aggregate = TUPTMultisig.aggregate_loci(public_loci, m_required=2)
print(f"\n2-of-4 Aggregate Locus: {aggregate}")
print(f"In TUPT space: 0 ≤ {aggregate} < {TUPT_MODULO} ✓")

# Verify threshold with simulated signatures
sigs = [signers[0].key, signers[1].key, 0, 0]  # Only 2 of 4 signed
is_valid, count = TUPTMultisig.verify_threshold(sigs, public_loci, m_required=2)
print(f"\nThreshold verification: valid={is_valid}, signatures={count}/4 (need 2)")

## Summary

| Property | Result |
|----------|--------|
| BIP-32 compatibility | ✓ Full path derivation |
| Determinism | ✓ Same seed → same keys |
| Key space | $\mathbb{Z}_{12289}$ (TUPT-projected) |
| Hardened derivation | ✓ Index ≥ 0x80000000 |
| Multisig | ✓ M-of-N φ-weighted aggregation |

*Nexus Resonance Codex — Resonance-Path HD Wallet Protocol*